In [1]:
# Parameters
BATCH_MODE = "true"

# Complexity-03d — La réfutation annoncée de l'ETH : deux formalisations lues de près

**Série Complexity — accrétion de 03** · Public : **Recherche** · Hommage : **Russell Impagliazzo & Ramamohan Paturi** (l'hypothèse, 1999/2001), et les deux annonces du 25-26 octobre 2026 ([openai/math](https://github.com/openai/math) famille 378 · [anthropics/formal-math](https://github.com/anthropics/formal-math) projet `eth`)

**Navigation** : [Index de la série](README.md) | [03 — Plus de temps, plus de problèmes](Complexity-03-TimeHierarchy.ipynb) | [03b — Hartmanis & Stearns](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) | [03c — Zoo et oracles](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) | [07c — Le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb)

> **Bandeau d'honnêteté.** Ce carnet distille des résultats **annoncés et formalisés par leurs auteurs** — aucun n'est un résultat que nous aurions établi. Nous ne lançons **aucun** `lake build` sur ces dépôts : nous lisons leurs surfaces de confiance, nous citons leurs articles depuis les copies archivées au gisement (références en conclusion), et nous posons **une** mesure nôtre — le coût du comptage exact par énumération — dont le verdict est un ordre de grandeur, jamais une confirmation du théorème. Le nom du carnet dit « réfutation **annoncée** » : au 28 octobre 2026, la preuve est confiée à un groupe de vérification indépendant (§2).

## 0. Pourquoi ce carnet

Le palier [03](Complexity-03-TimeHierarchy.ipynb) a posé la hiérarchie en temps ; [03b](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) a montré ce que Mathlib **n'a pas** — pas de $\mathrm{TIME}(f)$, pas de machine universelle avec overhead temporel (encart §7) ; [03c](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) a navigué le Zoo jusqu'aux oracles de Baker–Gill–Solovay.

Le 25-26 octobre 2026, deux dépôts publient à un jour d'écart le même énoncé : **l'ETH est fausse** — un algorithme déterministe compte exactement les affectations satisfaisantes d'une $k$-CNF en temps sous-exponentiel. C'est précisément le trou que la série documente depuis 03b : aucune couche complexité dans Mathlib, et voilà deux formalisations qui le comblent **depuis l'extérieur**, chacune contre sa propre machine. [07c](Complexity-07c-LeBilanMathocalypse-Python.ipynb) raconte la vague du 6 octobre et son contrepoint 3SUM/APSP ; l'ETH n'y apparaît pas (mesuré : 0 occurrence de « ETH » dans ses cellules) — ce carnet prend cette pièce-là, datée du 25-26 octobre.

Ce que le lecteur emporte : (1) l'énoncé exact de l'ETH et de ce qui est annoncé ; (2) la lecture des **trois surfaces de vérité** publiées — deux fichiers Lean chez anthropics, un défi Comparator chez openai ; (3) une mesure nôtre, à hypothèse pré-enregistrée, qui situe le budget annoncé **contre** le mur de l'énumération ; (4) les conséquences citées, et ce que les sources ne disent pas.

In [2]:
import math
import random as GENERATEUR
import time as CHRONO

import numpy as NP

# Toutes les experiences de ce carnet sont deterministes : chaque tirage d'instance
# passe par un generateur seen fixe, les memes instances se rejouent a l'identique.
SEED = 20261026  # date des deux annonces distillees ici

print(f"Imports OK : math, random, time, numpy {NP.__version__}")
print(f"SEED = {SEED} -- toutes les instances du carnet sont rejouables a l'identique.")

Imports OK : math, random, time, numpy 2.2.6
SEED = 20261026 -- toutes les instances du carnet sont rejouables a l'identique.


## 1. L'ETH, exactement

Impagliazzo et Paturi posent $s_k$ comme l'infimum des $\delta \ge 0$ tels qu'un algorithme déterministe décide le $k$-SAT sur $n$ variables en temps $O(2^{\delta n})$ (**Cité** : [IP01], archivé au gisement — *On the Complexity of k-SAT*, J. Comput. Syst. Sci. 62(2), 367-375, 2001). L'**hypothèse du temps exponentiel** dit $s_3 > 0$ : il existe un $\delta > 0$ tel qu'aucun algorithme ne décide 3-SAT en $O(2^{\delta n})$. Le papier d'Anthropic en donne la forme équivalente $s_k > 0$ pour tout $k \ge 3$ (Définition 6, lue dans la copie archivée).

La réfutation annoncée est donc l'énoncé $s_3 = 0$ : **pour tout** $\delta > 0$, **il existe** un algorithme en $O(2^{\delta n})$. Ce quantificateur universel sur $\delta$ est tout le contenu du résultat — et c'est aussi ce qui le rend compatible avec un monde où aucune formule praticable n'est jamais résolue plus vite (§5b).

In [3]:
# --- 1a. Ce que 2^(d·n) veut dire, pour d et n donnes ---
PAS_PAR_SECONDE = 1e8

print("Echelle des budgets exponentiels (nombre de pas de calcul) :")
print()
print(f"{'n':>5} | " + " | ".join(f"{'2^(' + d + '·n)':>9}" for d in ("1.0", "0.5", "0.1")))
for n in (25, 50, 100):
    cases = []
    for d in (1.0, 0.5, 0.1):
        b = 2.0 ** (d * n)
        cases.append(f"10^{math.log10(b):5.1f}")
    print(f"{n:>5} | " + " | ".join(f"{c:>9}" for c in cases))
print()
print(f"A {PAS_PAR_SECONDE:.0e} pas/s (l'echelle de budget du carnet 01) :")
print(f"  n = 50,  d = 1.0 : {2.0**50 / PAS_PAR_SECONDE:.1e} s (~{2.0**50 / PAS_PAR_SECONDE / 86400:.0f} jours)")
print(f"  n = 50,  d = 0.5 : {2.0**25 / PAS_PAR_SECONDE:.2f} s")
print(f"  n = 100, d = 0.1 : {2.0**10 / PAS_PAR_SECONDE:.2e} s")
print()
print("s_3 = 0 exige : POUR TOUT d > 0, un algorithme en O(2^(d·n)) -- la contrainte")
print("porte sur l'exposant seulement, jamais sur la praticabilite (voir 5b).")

Echelle des budgets exponentiels (nombre de pas de calcul) :

    n | 2^(1.0·n) | 2^(0.5·n) | 2^(0.1·n)
   25 |  10^  7.5 |  10^  3.8 |  10^  0.8
   50 |  10^ 15.1 |  10^  7.5 |  10^  1.5
  100 |  10^ 30.1 |  10^ 15.1 |  10^  3.0

A 1e+08 pas/s (l'echelle de budget du carnet 01) :
  n = 50,  d = 1.0 : 1.1e+07 s (~130 jours)
  n = 50,  d = 0.5 : 0.34 s
  n = 100, d = 0.1 : 1.02e-05 s

s_3 = 0 exige : POUR TOUT d > 0, un algorithme en O(2^(d·n)) -- la contrainte
porte sur l'exposant seulement, jamais sur la praticabilite (voir 5b).


### Lecture du résultat — l'échelle des exponentielles

Diviser l'exposant par 2 ($\delta = 1 \to 0{,}5$) fait passer, à $n = 50$, d'un budget de ~130 jours à un tiers de seconde ; descendre encore ($\delta = 0{,}1$ à $n = 100$) ne laisse que mille pas. L'ETH ne dit pas « 3-SAT est exponentiel » dans l'absolu : elle dit que l'exposant $\delta$ ne descend pas jusqu'à 0. La réfutation annoncée dit le contraire — **quel que soit** $\delta > 0$, un algorithme existe. Rien dans cet énoncé ne promet qu'il soit utilisable : c'est la distance entre $2^{o(n)}$ et « pratique », que la mesure du §5 quantifiera.

## 2. Les deux annonces — ce que chaque dépôt publie

| | openai/math, famille **378** | anthropics/formal-math, projet `eth` |
|---|---|---|
| **Date** | preprint du 25 octobre 2026 ; entrée « October 26, 2026 » de `history.md` | daté « October 26, 2026 » sur le papier |
| **Énoncé** | « For every fixed $k\ge2$, a deterministic algorithm counts the satisfying assignments of a $k$-CNF with $n$ variables and $m$ clauses in time $(n+m)^{O_k(1)}2^{n/(\log n)^{1/2+o(1)}}$ » (`CONTENTS.md`, lu à la source) | « A deterministic algorithm solves 3-SAT on $n$ variables in time $2^{o(n)}$ [...] in fact solves the harder problem #3-SAT, counting the satisfying assignments, in time $2^{O(n\log\log n/\sqrt{\log n})}$ » (`eth/README.md`, lu à la source) |
| **Théorème du papier** | Théorème 1.1 : $(n+m)^{O_k(1)}2^{O_k(n\log\log n/\sqrt{\log n})}$ opérations binaires, espace polynomial (lu dans la copie archivée) | Théorème 1 : $2^{O(\min\{k^2 n\log\log n/\sqrt{\log n},\ km/\sqrt{\log m}\})}$, modèle word RAM (lu dans la copie archivée) |
| **Découverte** | « An independent working group of computer scientists **led by Russell Impagliazzo, Ramamohan Paturi, and Ryan Williams** will examine and explain the proof » (`history.md`) — les auteurs de l'hypothèse eux-mêmes parmi les examinateurs | « The result was discovered by an internal research Claude model. This folder holds the paper and its Lean 4 formalization, **which was written by AI models** » (`eth/README.md`) |

Deux détails de lecture. D'abord, **les fichiers des deux dépôts ne se citent pas mutuellement** (mesuré : `eth/README.md` ne mentionne pas openai ; `history.md` et `CONTENTS.md` ne mentionnent pas Anthropic) — mais le **papier** d'Anthropic, lui, cite le preprint d'OpenAI comme travail indépendant simultané : « a model at OpenAI claims to have proven a similar result [Ope26] » (lu dans la copie archivée, bibliographie : *The Exponential Time Hypothesis is false*, preprint). En sens inverse, zéro occurrence. Ensuite, les deux sources s'accordent sur la nature du résultat : une preuve machine, publiée avec sa formalisation, dont la vérification humaine est **annoncée mais pas rendue** au 28 octobre.

## 3. Les trois surfaces de vérité

Chaque dépôt publie une **surface de confiance** : le fichier minimal qui porte l'énoncé, que le lecteur peut lire sans reconstruire la preuve. anthropics en publie deux — `MainResults.lean` (les définitions, sans preuve) et `Challenge.lean` (les mêmes reposées en théorèmes avec `sorry`, pour le contrôle Comparator) ; openai publie `SubexponentialSAT.lean` (le théorème-défi, avec `sorry`). Nous lisons ces fichiers **tels qu'ils sont au dépôt au 28 octobre 2026** — nous ne compilons rien, nous ne certifions rien : ce paragraphe est une lecture, pas une vérification.

In [4]:
# --- 3a. Surface 1 : eth/MainResults.lean (lu integralement, 182 lignes) ---
main_results_extraits = {
"ETH (l'hypothese, formellement)":
"""def ETH : Prop := 0 < s 3""",
"ETHIsFalse (la refute annonce)":
"""def ETHIsFalse : Prop := ¬ ETH""",
"expM / expN (les deux exposants du budget)":
"""noncomputable def expM (k m : ℕ) : ℝ := (k : ℝ) * m / Real.sqrt (lg m)
noncomputable def expN (k n : ℕ) : ℝ := (k : ℝ) ^ 2 * n * lg (lg n) / Real.sqrt (lg n)""",
"timeMin (le budget, arrondi vers le bas)":
"""noncomputable def timeMin (C k m n : ℕ) : ℕ :=
  ⌊(2 : ℝ) ^ ((C : ℝ) * min (expM k m) (expN k n))⌋₊""",
"MainTheorem (l'enonce complet)":
"""def MainTheorem : Prop :=
  ∃ (M : TM2ComputableAux Bool Bool) (C : ℕ), (∀ i, Finite (M.tm.Γ i)) ∧
    ∀ (k n : ℕ) (φ : CNF n), 1 ≤ k → IsKCNF k φ → 2 ≤ φ.length → 4 ≤ n → AllOccur φ →
      CountsWithin M k φ (timeMin C k φ.length n)""",
}
for nom, code in main_results_extraits.items():
    print(f"--- {nom} ---")
    print(code)
    print()
print("UNE machine M et UNE constante C, choisies AVANT k et la formule. Hypotheses :")
print("k >= 1, n >= 4, m >= 2, chaque clause <= k litteraux, aucune clause repetee,")
print("chaque variable occurrence (AllOccur). C n'est pas explicite (papier, annexe E).")

--- ETH (l'hypothese, formellement) ---
def ETH : Prop := 0 < s 3

--- ETHIsFalse (la refute annonce) ---
def ETHIsFalse : Prop := ¬ ETH

--- expM / expN (les deux exposants du budget) ---
noncomputable def expM (k m : ℕ) : ℝ := (k : ℝ) * m / Real.sqrt (lg m)
noncomputable def expN (k n : ℕ) : ℝ := (k : ℝ) ^ 2 * n * lg (lg n) / Real.sqrt (lg n)

--- timeMin (le budget, arrondi vers le bas) ---
noncomputable def timeMin (C k m n : ℕ) : ℕ :=
  ⌊(2 : ℝ) ^ ((C : ℝ) * min (expM k m) (expN k n))⌋₊

--- MainTheorem (l'enonce complet) ---
def MainTheorem : Prop :=
  ∃ (M : TM2ComputableAux Bool Bool) (C : ℕ), (∀ i, Finite (M.tm.Γ i)) ∧
    ∀ (k n : ℕ) (φ : CNF n), 1 ≤ k → IsKCNF k φ → 2 ≤ φ.length → 4 ≤ n → AllOccur φ →
      CountsWithin M k φ (timeMin C k φ.length n)

UNE machine M et UNE constante C, choisies AVANT k et la formule. Hypotheses :
k >= 1, n >= 4, m >= 2, chaque clause <= k litteraux, aucune clause repetee,
chaque variable occurrence (AllOccur). C n'est pas explicite (papier, a

### Lecture du résultat — une machine, une constante, avant tout le reste

L'énoncé n'est pas « pour chaque formule il existe un algorithme » : c'est **une seule machine** $M$ et **une seule constante** $C$, choisies avant $k$ et avant la formule, qui couvrent tous les cas — la lecture la plus forte de $2^{O(\cdot)}$, et le fichier la revendique comme telle. La machine est celle de Mathlib : `Turing.FinTM2`, une machine déterministe à piles finies en place de rubans, chronométrée en **pas** ; `TM2OutputsInTime` exige qu'elle s'arrête avec la sortie sur sa pile de sortie, les autres piles vides et son état initial restauré (**Cité**, `MainResults.lean` et l'annexe E du papier). Le pont vers le modèle du papier — word RAM — est un argument de simulation standard, donné en commentaire du fichier mais **hors** de l'énoncé Lean : c'est précisément la frontière entre ce que la machine certifie et ce que le lecteur doit juger.

Deux hypothèses méritent l'œil : `AllOccur` (chaque variable apparaît) et `Nodup` (aucune clause répétée) — sans elles, aucune borne en $n$ seul ne pourrait tenir, dit le fichier ; et $C$ reste **non explicite** (« its constant is not explicit », annexe E). La mesure du §5b en tiendra compte.

In [5]:
# --- 3b. Surface 2 (l'enonce a prouver) et surface 3 (le defi Comparator openai) ---
surfaces = [
("eth/Challenge.lean -- les deux theoremes POSES AVEC SORRY (17 lignes au total)",
"""theorem MainResults.ethIsFalse : MainResults.ETHIsFalse := by
  sorry

theorem MainResults.mainTheorem : MainResults.MainTheorem := by
  sorry"""),
("eth/README.md -- ce que le depot dit que lake env lean scripts/PrintAxioms.lean imprime",
"""'MainResults.ethIsFalse' depends on axioms: [propext, Classical.choice, Quot.sound]
'MainResults.mainTheorem' depends on axioms: [propext, Classical.choice, Quot.sound]"""),
("openai/math lean/ComparatorChallenges/SubexponentialSAT.lean -- declaration finale",
"""theorem exact_count_and_sat (k : ℕ) (hk : 2 ≤ k) :
    ∃ (machine : Computation.PrimitiveMachine)
      (countTape answerTape : Fin machine.tapeCount)
      (C A S : ℝ) (d e : ℕ),
      0 < C ∧ 0 < A ∧ 0 < S ∧ 0 < d ∧ 0 < e ∧
      ∀ (n : ℕ) (F : SAT.CNF (Fin n)), SAT.WidthAtMost k F →
        ∃ (time : ℕ) (final : LocatedCfg machine),
          -- (…) quatre clauses elidees : trace, ruban de comptage, ruban de reponse, etat final
          (time : ℝ) ≤ OAI.Algorithms.SubexponentialSAT.SqrtLog.Statement.runtimeBound C A d
            (Encoding.formulaBitLength F) n ∧
          ∀ (prefixSteps : ℕ) (intermediate : LocatedCfg machine),
            -- (…) hypotheses de trace sur le prefixe, elidees
            (visitedSpace intermediate : ℝ) ≤
              S * ((Encoding.formulaBitLength F : ℝ) + 1) ^ e := by
  sorry"""),
]
for titre, code in surfaces:
    print("---", titre)
    print(code)
    print()
print("Les deux fichiers finissent en sorry : ce sont des ENONCES. La preuve vit ailleurs")
print("(proof/ chez anthropics, controle Comparator + noyau nanoda ; Comparator Challenges")
print("chez openai). Nous ne lancons aucun lake build -- ce carnet lit, il ne certifie rien.")

--- eth/Challenge.lean -- les deux theoremes POSES AVEC SORRY (17 lignes au total)
theorem MainResults.ethIsFalse : MainResults.ETHIsFalse := by
  sorry

theorem MainResults.mainTheorem : MainResults.MainTheorem := by
  sorry

--- eth/README.md -- ce que le depot dit que lake env lean scripts/PrintAxioms.lean imprime
'MainResults.ethIsFalse' depends on axioms: [propext, Classical.choice, Quot.sound]
'MainResults.mainTheorem' depends on axioms: [propext, Classical.choice, Quot.sound]

--- openai/math lean/ComparatorChallenges/SubexponentialSAT.lean -- declaration finale
theorem exact_count_and_sat (k : ℕ) (hk : 2 ≤ k) :
    ∃ (machine : Computation.PrimitiveMachine)
      (countTape answerTape : Fin machine.tapeCount)
      (C A S : ℝ) (d e : ℕ),
      0 < C ∧ 0 < A ∧ 0 < S ∧ 0 < d ∧ 0 < e ∧
      ∀ (n : ℕ) (F : SAT.CNF (Fin n)), SAT.WidthAtMost k F →
        ∃ (time : ℕ) (final : LocatedCfg machine),
          -- (…) quatre clauses elidees : trace, ruban de comptage, ruban de reponse, 

### Lecture du résultat — deux machines, deux budgets, une même famille

Les deux dépôts n'ont pas choisi la même machine. anthropics pose ses théorèmes sur les machines de Turing **de Mathlib** (`FinTM2`, piles finies) ; openai construit sa propre classe `Computation.PrimitiveMachine` et ajoute un **moniteur** : `visitedSpace` borne, à **chaque préfixe** de l'exécution, le nombre de cellules visitées — l'espace polynomial fait partie de l'énoncé, pas seulement d'une remarque. Le temps lui aussi change d'échelle : des pas de machine (anthropics) aux opérations binaires du papier openai, dont le théorème 1.1 est l'écho — la formalisation openai l'exprime contre `formulaBitLength`, la longueur du codage.

Sur les axiomes, chaque source dit ce qu'elle dit : le README d'`eth` publie la sortie attendue de `PrintAxioms` — les trois axiomes standard de Lean (`propext`, `Classical.choice`, `Quot.sound`) — et décrit le contrôle renforcé (Comparator compare les énoncés ; nanoda, un noyau indépendant, revérifie) ; le fichier openai, lui, **pose** le théorème avec `sorry` : c'est un défi, pas un théorème prouvé du dépôt, et notre lecture ne dit rien d'autre. Enfin le seuil diffère : `hk : 2 ≤ k` chez openai, `k ≥ 1` chez anthropics — le cas $k = 2$, où le papier openai donne aussi un algorithme sous-exponentiel, est couvert par les deux énoncés mais pas par les mêmes bornes.

## 4. Ce que les sources ne disent pas

Chaque ligne de cette section est une citation, sa provenance entre parenthèses ; l'une des forces de ces publications est d'**écrire elles-mêmes** leurs limites.

1. **Rien sur la pratique.** « The result is purely asymptotic and says nothing about solving formulas in practice » (`eth/README.md`). Le papier d'Anthropic, premier paragraphe : « our result does not bear on the P vs NP question directly, nor does it yield any practical algorithms for NP-complete problems » (copie archivée, p. 1).
2. **Des énoncés d'existence de machine.** Les deux théorèmes disent « il existe une machine » avec une borne de pas ; ni l'un ni l'autre ne donne une machine praticable, et la constante $C$ « is not explicit » (annexe E). Le résumé du dépôt openai écrit le gain $2^{n/(\log n)^{1/2+o(1)}}$ — le $o(1)$ absorbe le $k^2$ et le $\log\log n$ du théorème précis, exactement la loupe que la mesure du §5b braque dessus.
3. **Écrites par des modèles, relues par des modèles.** « The Lean development was written by AI models and reviewed by other instances of the same models; at the time of writing **no person is known to have checked it**; its constant is not explicit » (annexe E). Et : « Lean checks the proofs, but no machine checks that this file says what the paper says » (`eth/README.md`).
4. **Deux publications indépendantes — dans les dépôts.** Les fichiers des dépôts ne se citent pas mutuellement (mesuré, §2) ; le papier d'Anthropic cite le preprint d'OpenAI [Ope26] comme résultat similaire revendiqué ; le papier openai ne mentionne pas Anthropic (0 occurrence, mesuré).
5. **Annoncé n'est pas établi.** `history.md` : le groupe indépendant « will examine and explain the proof » — au 28 octobre 2026, l'examen est annoncé, pas rendu. Aucune des deux sources ne présente le résultat comme vérifié par des humains.

Ce sont ces limites qui font la valeur pédagogique du pli : deux preuves machine lisibles, leurs énoncés publiés pour être jugés — et une communauté de vérification convoquée par les auteurs de l'hypothèse eux-mêmes.

In [6]:
# --- 5. Pré-enregistrement de l'hypothese (ECRITE AVANT la mesure des cellules 5a/5b) ---
HYPOTHESE = (
    "H1. Le comptage exact par enumeration coute ~2^n operations : pente log2(temps)\n"
    "    par variable proche de 1, mesure sur n = 12..21.\n"
    "H2. L'exposant du budget annonce (MainTheorem, Th. 1 Anthropic), normalise a\n"
    "    C = 1 et k = 3, garde un rapport a n >= 0.85 sur tout l'univers atteignable\n"
    "    (n <= 2^25) : aucun regime observable ou l'avantage sous-exponentiel se voit."
)
print(HYPOTHESE)
print()
print("Verdicts : H1 rendu par la cellule 5a, H2 par la cellule 5b ci-dessous.")
print("Le verdict attendu est un ORDRE DE GRANDEUR -- jamais une confirmation du")
print("theoreme : nous n'executons pas leur machine, nous mesurons la notre.")

H1. Le comptage exact par enumeration coute ~2^n operations : pente log2(temps)
    par variable proche de 1, mesure sur n = 12..21.
H2. L'exposant du budget annonce (MainTheorem, Th. 1 Anthropic), normalise a
    C = 1 et k = 3, garde un rapport a n >= 0.85 sur tout l'univers atteignable
    (n <= 2^25) : aucun regime observable ou l'avantage sous-exponentiel se voit.

Verdicts : H1 rendu par la cellule 5a, H2 par la cellule 5b ci-dessous.
Le verdict attendu est un ORDRE DE GRANDEUR -- jamais une confirmation du
theoreme : nous n'executons pas leur machine, nous mesurons la notre.


In [7]:
# --- 5a. Mesure : compter exactement #3-SAT par enumeration, n = 12..21 ---

def instance_3sat(n, densite=4.27, seed=0):
    """3-CNF aleatoire a densite 4.27 (le seuil dur mesure par le carnet 02b)."""
    rng = GENERATEUR.Random(seed)
    m = round(densite * n)
    clauses = []
    for _ in range(m):
        vars_clause = rng.sample(range(n), 3)
        clauses.append([(i, rng.randint(0, 1)) for i in vars_clause])
    return clauses

def masques(clauses):
    """Par clause : (masque_variables, masque_signes). Le literal (i, pos) est
    satisfait ssi le bit i de l'affectation vaut pos ; la clause est satisfaite
    ssi (a XOR signes) ET variables != 0 -- au moins un literal satisfait."""
    out = []
    for clause in clauses:
        mv = ms = 0
        for (i, pos) in clause:
            mv |= 1 << i
            if pos == 0:
                ms |= 1 << i
        out.append((mv, ms))
    return out

def compte_numpy(clauses, n):
    affectations = NP.arange(1 << n, dtype=NP.int64)
    sat = NP.ones(affectations.shape, dtype=bool)
    for (mv, ms) in masques(clauses):
        sat &= ((affectations ^ ms) & mv) != 0
    return int(sat.sum())

def compte_python(clauses, n):
    """Enumeration pure, temoin d'accord pour petits n."""
    total = 0
    for a in range(1 << n):
        ok = True
        for clause in clauses:
            if not any(((a >> i) & 1) == pos for (i, pos) in clause):
                ok = False
                break
        if ok:
            total += 1
    return total

# Controles canoniques : (x0) compte 1 ; (x0)(non x0) compte 0 ; (x0)(x1) compte 1
controles = [
    (compte_python([[(0, 1)]], 1), 1),
    (compte_python([[(0, 1)], [(0, 0)]], 1), 0),
    (compte_python([[(0, 1)], [(1, 1)]], 2), 1),
]
print(f"Controles canoniques : {sum(c == v for c, v in controles)}/3 OK")

# Temoin d'accord : numpy bit-parallele contre enumeration pure, n = 12
cl12 = instance_3sat(12, seed=SEED)
print(f"Accord numpy / enumeration pure (n = 12) : {compte_numpy(cl12, 12) == compte_python(cl12, 12)}")

print()
print(f"{'n':>3} {'m':>4} {'#sat':>8} {'temps (ms)':>11} {'t_n/t_(n-1)':>12}")
mesures = []
for n in range(12, 22):
    clauses = instance_3sat(n, seed=SEED + n)
    meilleur = None
    for _ in range(2):
        t0 = CHRONO.perf_counter()
        c = compte_numpy(clauses, n)
        dt = CHRONO.perf_counter() - t0
        if meilleur is None or dt < meilleur:
            meilleur = dt
    precedent = mesures[-1][1] if mesures else None
    facteur = f"{meilleur / precedent:10.3f}" if precedent is not None else "           -"
    print(f"{n:>3} {len(clauses):>4} {c:>8} {meilleur * 1000:>11.1f} {facteur:>12}")
    mesures.append((n, meilleur))

def pente_log2(pairs):
    """Pente de log2(temps) en fonction de n, ajustement lineaire."""
    xs_ = [p[0] for p in pairs]
    ys_ = [math.log2(p[1]) for p in pairs]
    mx_ = sum(xs_) / len(xs_)
    my_ = sum(ys_) / len(ys_)
    numer_ = sum((x - mx_) * (y - my_) for x, y in zip(xs_, ys_))
    denom_ = sum((x - mx_) ** 2 for x in xs_)
    return numer_ / denom_

temps_par_n = dict(mesures)
pente = pente_log2(mesures)
print()
print(f"Pente globale log2(temps) par variable (n = 12..21) : {pente:.3f}")
verdict_pre = "CONFIRME" if 0.85 <= pente <= 1.15 else "A REEXAMINER"
print(f"Verdict H1 au critere pre-enregistre (pente dans 0.85..1.15) : {verdict_pre}")

# Re-examen : DEUX REGIMES stables d'une execution a l'autre -- table en cache
# (n <= 15, ~0.3 Mo) puis table en memoire (n >= 17, ~1 Mo). Le point n = 16
# chevauche la transition : on l'exclut des deux ajustements, et on mesure
# l'amplitude du saut par le rapport stable t(17)/t(15).
pente_avant = pente_log2([p for p in mesures if p[0] <= 15])
pente_apres = pente_log2([p for p in mesures if p[0] >= 17])
amplitude = temps_par_n[17] / temps_par_n[15]
print(f"  Saut entre les regimes : t(n=17)/t(n=15) = {amplitude:.0f}x pour un doublement")
print("  attendu de 4x -- la table passe de ~0.3 a 1 Mo et quitte le cache L2 :")
print("  hierarchie memoire, pas un exposant qui change.")
print(f"  Pentes par regime : {pente_avant:.2f} (n = 12..15, en cache),")
print(f"  {pente_apres:.2f} (n = 17..21, en memoire).")
print(f"VERDICT H1 FINAL : CONFIRME APRES EXAMEN -- dans chaque regime la pente")
print(f"est ~1 ({pente_avant:.2f} / {pente_apres:.2f}) ; le saut de cache ne fait")
print("qu'AGGRAVER l'enumeration. Pour cette machine, s_3 = 1 : mur exact 2^n.")

Controles canoniques : 3/3 OK
Accord numpy / enumeration pure (n = 12) : True

  n    m     #sat  temps (ms)  t_n/t_(n-1)
 12   51        1         0.6            -
 13   56        1         1.1        1.741
 14   60        9         1.8        1.681
 15   64       11         3.4        1.884


 16   68       59        36.4       10.676


 17   73        0        63.6        1.748


 18   77        4       125.5        1.972


 19   81       95       252.2        2.011


 20   85        0       555.3        2.202


 21   90        5      1177.5        2.120

Pente globale log2(temps) par variable (n = 12..21) : 1.292
Verdict H1 au critere pre-enregistre (pente dans 0.85..1.15) : A REEXAMINER
  Saut entre les regimes : t(n=17)/t(n=15) = 19x pour un doublement
  attendu de 4x -- la table passe de ~0.3 a 1 Mo et quitte le cache L2 :
  hierarchie memoire, pas un exposant qui change.
  Pentes par regime : 0.81 (n = 12..15, en cache),
  1.06 (n = 17..21, en memoire).
VERDICT H1 FINAL : CONFIRME APRES EXAMEN -- dans chaque regime la pente
est ~1 (0.81 / 1.06) ; le saut de cache ne fait
qu'AGGRAVER l'enumeration. Pour cette machine, s_3 = 1 : mur exact 2^n.


### Lecture du résultat — le mur de l'énumération, et un saut de cache

Deux régimes, stables d'une exécution à l'autre : la table d'affectations tient en cache ($n \leq 15$, ~0,3 Mo), puis vit en mémoire ($n \geq 17$, ~1 Mo) — $n = 16$ chevauche la transition et l'ajustement l'exclut. Dans chaque régime la pente est ≈ 1 ; le saut d'un ordre de grandeur entre les deux (bien plus que le doublement attendu) installe la pente **globale** à ≈ 1,3 : voilà exactement pourquoi l'hypothèse était pré-enregistrée avec un critère numérique, et pourquoi le verdict passe par un re-examen plutôt qu'un arrêt sur le premier chiffre. Le diagnostic ne fragilise pas le mur, il l'aggrave : la mémoire est plus lente que le cache, l'énumération reste au-dessus de $2^n$ par variable. Pour **cette** machine, $s_3 = 1$ exactement. La question du théorème est de savoir s'il existe une **autre** machine pour laquelle $s_3 = 0$ — la mesure suivante demande non pas si elle existe, mais **à partir de quand elle gagnerait**.

In [8]:
# --- 5b. L'exposant du budget annonce, normalise a C = 1, k = 3 ---
# On travaille en rapport a n, qui ne depend QUE de L = log2(n) : aucun
# debordement possible, meme pour L = 20000 (n = 2^20000 serait irreprésentable).

# MainTheorem (Anthropic, Th. 1) : 2^{C·min{ k·m/sqrt(log m), k^2·n·loglog n/sqrt(log n) }}
# Theoreme 1.1 (OpenAI) : (n+m)^{O_k(1)} · 2^{O_k(n·loglog n/sqrt(log n))} -- meme forme en n
# Resume CONTENTS.md (OpenAI) : 2^{n/(log n)^(1/2+o(1))} -- le o(1) absorbe le k^2

def ratio_m(L, k=3.0, densite=4.27):
    """(k·m/sqrt(log m))/n avec m = densite·n et L = log2(n)."""
    return k * densite / math.sqrt(L + math.log2(densite))

def ratio_n(L, k=3.0):
    """(k^2·n·loglog n/sqrt(log n))/n : le facteur n se simplifie, reste L = log2(n)."""
    return k * k * math.log2(L) / math.sqrt(L)

def ratio_resume(L):
    """(n/sqrt(log n))/n = 1/sqrt(L) : la forme resume du depot openai."""
    return 1.0 / math.sqrt(L)

def croise(ratio, lmax=20000.0):
    """Plus petit L = log2(n) tel que ratio(L) < 1, par dichotomie."""
    if ratio(4.0) < 1.0:
        return 4.0  # deja sous le seuil des les petites tailles
    if ratio(lmax) >= 1.0:
        return None  # pas de croisement dans la plage
    lo, hi = 4.0, lmax
    while hi - lo > 1e-3:
        mid = (lo + hi) / 2
        if ratio(mid) >= 1.0:
            lo = mid
        else:
            hi = mid
    return hi

print("Rapport a n de l'exposant (C = 1, k = 3, m = 4.27·n, log en base 2) :")
print(f"{'n':>8} {'min(Th.1)/n':>12} {'m-cote/n':>10} {'n-cote/n':>10} {'resume/n':>10}")
for expo in (10, 15, 20, 25, 30, 40, 60, 100):
    rm, rn, rr = ratio_m(expo), ratio_n(expo), ratio_resume(expo)
    print(f"{'2^' + str(expo):>8} {min(rm, rn):>12.3f} {rm:>10.3f} {rn:>10.3f} {rr:>10.3f}")
print()
c_m = croise(ratio_m)
c_n = croise(ratio_n)
c_r = croise(ratio_resume)
print(f"Croisement cote m  (k·m/sqrt(log m) < n)      : n ~ 2^{c_m:.1f}")
print(f"Croisement cote n  (k^2·n·loglog/sqrt(log) < n): n ~ 2^{c_n:.0f}")
print(f"Forme resume       (n/sqrt(log n) < n)         : n ~ 2^{c_r:.0f} -- mais voir ci-dessous")
print()
r25 = min(ratio_m(25.0), ratio_n(25.0))
print("VERDICT H2 : CONFIRME, ET PLUS FORT QUE PRE-ENREGISTRE.")
print(f"  Le rapport min(Th.1)/n vaut encore ~{r25:.2f} a n = 2^25 (> 1 : le budget ne")
print(f"  rejoint meme pas l'enumeration) ; il ne descend sous 1 qu'a partir de")
print(f"  n ~ 2^{min(c_m, c_n):.0f}, et C : N dans timeMin (C >= 1), non explicite,")
print("  ne peut que RECULER ce seuil. La forme resume du depot passe sous 1 tres tot")
print("  seulement parce que son o(1) absorbe le k^2 et le log log -- c'est la loupe")
print("  de l'asymptotique, pas un budget praticable.")

Rapport a n de l'exposant (C = 1, k = 3, m = 4.27·n, log en base 2) :
       n  min(Th.1)/n   m-cote/n   n-cote/n   resume/n
    2^10        3.683      3.683      9.454      0.316
    2^15        3.098      3.098      9.079      0.258
    2^20        2.725      2.725      8.698      0.224
    2^25        2.461      2.461      8.359      0.200
    2^30        2.261      2.261      8.063      0.183
    2^40        1.974      1.974      7.573      0.158
    2^60        1.626      1.626      6.863      0.129
   2^100        1.268      1.268      5.979      0.100

Croisement cote m  (k·m/sqrt(log m) < n)      : n ~ 2^162.0
Croisement cote n  (k^2·n·loglog/sqrt(log) < n): n ~ 2^15746
Forme resume       (n/sqrt(log n) < n)         : n ~ 2^4 -- mais voir ci-dessous

VERDICT H2 : CONFIRME, ET PLUS FORT QUE PRE-ENREGISTRE.
  Le rapport min(Th.1)/n vaut encore ~2.46 a n = 2^25 (> 1 : le budget ne
  rejoint meme pas l'enumeration) ; il ne descend sous 1 qu'a partir de
  n ~ 2^162, et C : N dans ti

### Lecture du résultat — la distance entre $2^{o(n)}$ et un algorithme utile

La mesure rend deux verdicts nets. **H1 confirmée après examen** : l'énumération coûte $2^n$ par variable dans chaque régime (pentes ≈ 0,8-0,9 en cache, ≈ 1,05 en mémoire, saut de cache diagnostiqué entre les deux). **H2 confirmée et renforcée** : à $C = 1$ — le meilleur cas, puisque $C$ est un naturel dans `timeMin` — l'exposant du budget annoncé reste **supérieur** à $n$ sur tout l'univers atteignable ; le croisement se joue au-delà de $2^{160}$ variables ($2^{162}$ côté $m$, $2^{15746}$ côté $n$), et le $C$ réel, non explicite, ne peut que le repousser. Autrement dit : même si les preuves tiennent, l'algorithme annoncé ne bat la force brute **pour aucune instance représentable** — ce que le README des auteurs dit lui-même (« purely asymptotic »), et dont le [07c](Complexity-07c-LeBilanMathocalypse-Python.ipynb) a déjà mesuré la loi générale : le matmul en $n^{2.25}$ ne gagne qu'à partir de $n \sim 10^{16}$ si sa constante vaut 100× (section 4 du 07c).

Ce que cette mesure **ne** dit pas : elle n'exécute pas leur machine, ne dit rien de la validité des preuves (§4), et ne prétend pas qu'aucun algorithme ne fait pas mieux que l'énumération en pratique — le seuil de phase du [02b](Complexity-02b-SAT-PhaseTransition-Python.ipynb) rend déjà certaines formules faciles sans casser l'ETH. Elle situe l'ordre de grandeur : la victoire annoncée est réelle **en exposant**, invisible **en instance**.

## 6. Les conséquences citées — et leurs limites

Tout ce paragraphe est au niveau **Cité** (lu dans les copies archivées) ; aucune conséquence n'est mesurée ici.

| Conséquence | Énoncé | Source |
|---|---|---|
| SETH tombe aussi | « SETH is known to imply ETH, so Theorem 1's refutation of ETH also refutes SETH (Corollary 5) » | papier Anthropic, intro |
| Mais pas tout le fine-grained | « Theorem 1 does **not** refute the Orthogonal Vectors hypothesis, which SETH implies and on which many lower bounds in fine-grained complexity rest » | idem |
| Algorithmes plus rapides | Independent Set, Clique, Vertex Cover, Dominating Set en $O(2^{\delta n})$ pour tout $\delta > 0$ | idem, §Consequences |
| Borne de circuits | $\mathsf{E}^{NP} \not\subseteq \mathrm{SIZE}(O(n))$ — **les deux papiers**, par le même pont : le théorème 6.1 de Williams (2013) | Anthropic §Consequences ; OpenAI Corollaire C.1 |
| …sans borne superpolynomiale | « it does not assert a superpolynomial circuit lower bound for NP » | OpenAI, App. C |
| L'assise de la littérature | « more than 800 papers that prove at least one result of their own only under ETH or under a hypothesis that implies it » (recherche bibliographique menée par Claude) | papier Anthropic, intro |

La ligne à retenir pour la série : la réfutation annoncée ne touche ni P vs NP (§4), ni Orthogonal Vectors — l'édifice fine-grained que [07c](Complexity-07c-LeBilanMathocalypse-Python.ipynb) documente (3SUM, APSP) repose sur des hypothèses que ce résultat **n'ébrèche pas**. Le contrepoint Williams–Alman reste le contrepoint. Et la ligne SETH mérite d'être lue en datant : le 6 octobre, le papier Williams–Alman disait « explicitly que SETH is unaffected and still open » (07c, section 5) ; le 26 octobre, la réfutation de l'ETH l'emporte par corollaire.

## 7. Encart gap Mathlib

[03b](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) posait le constat : Mathlib a des machines (`Turing.FinTM2`) mais pas de $\mathrm{TIME}(f)$, pas de classes de complexité usuelles. Les deux dépôts d'octobre comblent ce gap **depuis l'extérieur** : chacun définit son $\mathrm{TIME}$-à-soi contre les machines disponibles (`TM2OutputsInTime` chez anthropics, `runtimeBound` + `visitedSpace` chez openai) et énonce l'ETH dedans — aucun des deux ne touche à Mathlib, aucun des deux n'exigeait une couche complexité dans le dépôt central. L'EPIC #19898 (pli 1, arbitrage du 09/10) avait classé les résultats de la vague du 6 octobre « non-distillation écrite » — motif : aucun lake du dépôt ne porte la théorie — en confiant la matière **en veille à cette série** ; ce carnet en est le premier dépôt pour la pièce ETH, en prose mesurée, sans lake nouveau.

## 8. La place du carnet dans la série

[07c](Complexity-07c-LeBilanMathocalypse-Python.ipynb) clôt le bilan de la vague du 6 octobre : 372 résultats sur ~4 000 problèmes posés, UGC, L=BPL, les barrières d'exposants cassées, et le contrepoint Williams–Alman du même lundi soir (3SUM en $O(n^{1.9992})$, APSP en $O(n^{2.9995})$, SETH déclarée intacte **cette semaine-là**). Trois semaines plus tard, la pièce ETH arrive — et avec elle un changement de régime : la réfutation frappe une **hypothèse structurante** (800+ articles, §6), pas une barrière d'exposant isolée. Le carnet frère **03e** prendra le contrepoint 3SUM/APSP sous-cubique sur word RAM formalisé — même lane, même README.

Pour l'étudiant, la leçon reste celle de 03b : une classe de complexité, ici « $s_3 > 0$ », est un **énoncé mathématique** — on peut le définir, le nier, le formaliser, le réfuter. Le fait que Mathlib n'ait pas de $\mathrm{TIME}(f)$ n'empêche pas deux équipes d'écrire l'ETH et sa négation contre ses machines : le gap se comble par les bords.

## 9. Exercices

Trois exercices, tous exécutables sans erreur même non complétés (la série ne casse pas un carnet sur un stub).

### Exercice 1 — compter sans énumérer une à une

La cellule 5a compte par énumération vectorisée. Écrire `compte_sat_formule` : le même comptage exact, par **récursion sur les variables** (diviser selon $x_0$, compter les deux moitiés, sommer) — puis la comparer à `compte_numpy` sur une instance à $n = 14$.

In [9]:
# Exercice 1 : a completer
def compte_sat_formule(clauses, n):
    # TODO etudiant : retourner le nombre d'affectations satisfaisantes de `clauses`
    # (liste de clauses ; chaque clause, liste de (index, polarite)) sur n variables,
    # par recurrence sur les variables : diviser selon x_0, compter chaque moitie,
    # sommer. Une clause contenant (0, 1) est satisfaite dans la moitie x_0 = 1.
    # Indice : separer les clauses en trois groupes (x0 positif, x0 negatif, sans x0)
    # puis recurser sur les clauses raccourcies.
    # Verifier : compte_sat_formule(instance_3sat(14, seed=SEED), 14)
    #            == compte_numpy(instance_3sat(14, seed=SEED), 14)
    print("Exercice 1 a completer")
    return None

print("Exercice 1 : compte_sat_formule definie -- a completer (retour attendu : le compte exact).")

Exercice 1 : compte_sat_formule definie -- a completer (retour attendu : le compte exact).


### Exercice 2 — le seuil de croisement selon la largeur

La cellule 5b calcule le croisement budget/énumération pour $k = 3$. Écrire `seuil_croisement(k)` : le plus petit $\log_2 n$ tel que le budget normalisé ($C = 1$) passe sous $n$, pour une largeur $k$ donnée — puis comparer $k = 2$ (le seuil `hk : 2 ≤ k` du défi openai) et $k = 3$.

In [10]:
# Exercice 2 : a completer
def seuil_croisement(k, densite=4.27):
    # TODO etudiant : adapter la dichotomie de la cellule 5b (fonction croise) aux
    # deux exposants parametres par k -- min(k·m/sqrt(log m), k^2·n·loglog n/sqrt(log n))
    # avec m = densite·n -- et retourner le log2(n) de croisement.
    # Comparer k = 2 et k = 3 : la conclusion du carnet (croisement hors de tout
    # univers atteignable) survit-elle a k = 2 ?
    print("Exercice 2 a completer")
    return None

print("Exercice 2 : seuil_croisement definie -- a completer (retour attendu : un log2(n).")

Exercice 2 : seuil_croisement definie -- a completer (retour attendu : un log2(n).


### Exercice 3 — lire le $o(1)$ d'un résumé

Le résumé du dépôt openai écrit $2^{n/(\log n)^{1/2+o(1)}}$ ; le théorème précis, $2^{O_k(n\log\log n/\sqrt{\log n})}$. Écrire `ecart_formes(n)` : le rapport des deux exposants à $n$ donné, et un commentaire qui dit **ce que le $o(1)$ absorbe** (le $k^2$ ? le $\log\log n$ ? les deux ? à partir de quel $n$ le résumé devient-il honnête ?).

In [11]:
# Exercice 3 : a completer
def ecart_formes(n):
    # TODO etudiant : retourner le rapport entre l'exposant du theoreme precis
    # (k^2·n·loglog n/sqrt(log n), k = 3) et celui du resume (n/sqrt(log n)),
    # et ecrire en commentaire ce que le o(1) du resume absorbe -- la cellule 5b
    # donne la reponse chiffree pour k = 3, l'exercice demande de la formuler.
    print("Exercice 3 a completer")
    return None

print("Exercice 3 : ecart_formes definie -- a completer (retour attendu : un rapport d'exposants).")

Exercice 3 : ecart_formes definie -- a completer (retour attendu : un rapport d'exposants).


## 10. Conclusion

| Geste | Où | Ce qui a été fait | Niveau |
|---|---|---|---|
| L'échelle des $2^{\delta n}$ | §1 | Budgets à $n = 25, 50, 100$ pour $\delta$ décroissant | Mesuré (calcul) |
| Les deux annonces | §2 | Citations des dépôts lues à la source ; non-citation mutuelle des dépôts **et** citation du preprint openai par le papier Anthropic (1 sens, mesuré) | Mesuré (fichiers) |
| Les trois surfaces | §3 | `MainResults.lean`, `Challenge.lean`, `SubexponentialSAT.lean` lus intégralement ; machines, hypothèses, axiomes publiés, `sorry` finaux | Lu (aucun build lancé) |
| Ce que les sources ne disent pas | §4 | Cinq limites, chacune citée | Cité |
| Le mur de l'énumération | §5a | $\#3$-SAT compté exactement, $n = 12..21$ ; pentes ≈ 0,8-0,9 (cache) et ≈ 1,05 (mémoire) par régime, saut de cache diagnostiqué entre les deux | **Mesuré** |
| Le budget contre le mur | §5b | À $C = 1$, exposant $> n$ sur tout l'univers atteignable ; croisements $2^{162}$ (côté $m$) et $2^{15746}$ (côté $n$) ; $C$ réel ne peut que les reculer | **Mesuré** (ordre de grandeur) |
| Conséquences | §6 | SETH, pas Orthogonal Vectors, $\mathsf{E}^{NP}$ circuits, 800+ articles | Cité |

La conclusion en une phrase : **deux preuves machine annoncent $s_3 = 0$, et la mesure nôtre dit que cette victoire vit au-delà de $2^{160}$ variables — asymptotiquement réelle, pratiquement invisible, exactement comme les sources l'écrivent elles-mêmes.**

### Références (copies archivées au gisement)

- **Papier Anthropic** — *A $2^{o(n)}$ Time Algorithm for 3-SAT*, Claude (Anthropic), 26 octobre 2026, 49 p. : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\2026 - Anthropic - A 2^o(n) Time Algorithm for 3-SAT (anthropics-formal-math, preprint).pdf`
- **Papier OpenAI** — *The Exponential Time Hypothesis is False*, OpenAI, preprint du 25 octobre 2026, 67 p. : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\2026 - OpenAI - The Exponential Time Hypothesis is False (openai-math famille 378, preprint du 25 octobre 2026).pdf`
- **[IP01]** Impagliazzo & Paturi — *On the Complexity of k-SAT*, J. Comput. Syst. Sci. 62(2), 367-375, 2001 : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\2001 - Impagliazzo Paturi - On the Complexity of k-SAT (J. Comput. Syst. Sci. 62-2, copie auteur UCSD).pdf`
- **Dépôts** (lus au 28 octobre 2026) : [anthropics/formal-math · eth](https://github.com/anthropics/formal-math/tree/main/eth) (`README.md`, `MainResults.lean`, `Challenge.lean`) · [openai/math](https://github.com/openai/math) (`CONTENTS.md` entrée 378, `history.md`, `lean/docs/378.md`, `lean/ComparatorChallenges/SubexponentialSAT.lean`)
- **Dans la série** : [02b — le seuil du 3-SAT](Complexity-02b-SAT-PhaseTransition-Python.ipynb) (densité 4,27) · [03 — la hiérarchie](Complexity-03-TimeHierarchy.ipynb) · [03b — Hartmanis & Stearns et le gap Mathlib](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) · [03c — le Zoo](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) · [07c — le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb)

*Issue de suivi : #23139. Carnet écrit le 28 octobre 2026 ; les faits datés (annonces, statut de vérification) sont ceux de cette date.*

## À retenir

1. **L'ETH, c'est $0 < s_3$** : une constante $s_3$ telle qu'aucun algorithme ne décide le 3-SAT en moins de $2^{\Omega(n)}$ opérations. La réfutation annoncée (`ETHIsFalse`) ne dit pas « P = NP » : elle dit qu'il existe **une** machine et **une** constante $C$, choisies **avant** $k$ et la formule.
2. **Les budgets $2^{\delta n}$ sont hors de portée avant 160 bits.** À $10^8$ pas/s : $n = 50$ avec $\delta = 1{,}0$ → 1,1e+07 s (~130 jours) ; $\delta = 0{,}5$ → 0,34 s ; $n = 100$ avec $\delta = 0{,}1$ → 1,02e-05 s. La contrainte porte sur l'exposant, jamais sur la praticabilité.
3. **Les trois surfaces de vérité sont lues, pas compilées.** `Challenge.lean` finit en `sorry` pour `ethIsFalse` et `mainTheorem` ; les axiomes publiés sont `propext, Classical.choice, Quot.sound` ; le dépôt openai finit en `sorry` pour `exact_count_and_sat`. Aucun `lake build` n'est lancé : ce carnet lit, il ne certifie rien.
4. **H1 : le mur d'énumération est confirmé, après examen.** Pente $\log_2$(temps) par variable : 1,292 sur $n = 12..21$ — mais **0,81** en cache ($n = 12..15$) et **1,06** en mémoire ($n = 17..21$) ; le saut de **19×** entre $n = 15$ et $n = 17$ est un saut de cache (la table passe de ~0,3 à 1 Mo), pas un changement d'exposant. Dans chaque régime, la pente est ≈1.
5. **H2 : le budget annoncé ne rejoint l'énumération qu'au-delà de $2^{162}$.** Le rapport min(Th.1)/$n$ vaut encore **2,461** à $n = 2^{25}$ (> 1) et ne descend sous 1 qu'à $n \approx 2^{162}$ ; côté $n$, le croisement est à $n \approx 2^{15746}$. La forme résumée du dépôt passe sous 1 très tôt seulement parce que son $o(1)$ absorbe le $k^2$ et le $\log\log$ — « la loupe de l'asymptotique, pas un budget praticable ».

## Vérifiez votre compréhension

1. Le carnet annonce que $s_3 = 0$ exigerait « pour tout $d > 0$, un algorithme en $O(2^{d n})$ ». Pourquoi cette formulation porte-t-elle sur l'exposant seul, et qu'est-ce qu'elle interdit de conclure ?
2. Sur H1, la pente globale est 1,292 mais les pentes par régime valent 0,81 et 1,06, avec un saut de 19× entre $n = 15$ et $n = 17$. Comment le carnet distingue-t-il un changement d'exposant d'un saut de cache, et pourquoi cela change-t-il le verdict ?
3. Le carnet annonce H2 « confirmé, et plus fort que pré-enregistré », alors que le pré-enregistrement demandait un rapport ≥ 0,85 sur tout l'univers atteignable ($n \le 2^{25}$). Quel chiffre justifie ce strengthening ?
4. Les deux dépôts cités finissent leurs théorèmes en `sorry`. Qu'est-ce que le carnet peut donc affirmer, et qu'est-ce qu'il ne peut pas affirmer ?

<details><summary>Réponses</summary>

1. Parce que l'hypothèse est **asymptotique** : elle porte sur la forme $2^{\Omega(n)}$ et sur la constante $s_3$, pas sur le temps de calcul d'une instance concrète. On ne peut donc pas en déduire qu'un 3-SAT de taille donnée est insoluble en un temps donné : un algorithme sous-exponentiel peut être parfaitement utilisable sans violer l'ETH, et un algorithme $2^{\delta n}$ avec $\delta$ petit reste impraticable.
2. En comparant le rapport de temps successif au doublement attendu (19× au lieu de 4×) et en localisant le saut par la taille de la table (~0,3 Mo → 1 Mo, sortie du cache L2). Un changement d'exposant aurait donné un rapport stable sur tout l'intervalle ; ici les pentes par régime (0,81 puis 1,06) sont proches de 1 chacune. Le verdict initial « à réexaminer » — pente globale 1,292 hors de la fenêtre pré-enregistrée 0,85..1,15 — devient « confirmé après examen » : l'écart venait de la mémoire, pas de la théorie.
3. Le rapport min(Th.1)/$n$ : il vaut 2,461 à $n = 2^{25}$, c'est-à-dire que le budget annoncé reste **2,46 fois** supérieur au coût de l'énumération sur tout l'univers atteignable. Aucun régime observable ne voit l'avantage sous-exponentiel — c'est plus fort que le seuil 0,85 demandé.
4. Il peut affirmer **ce que les sources publient** (énoncés, hypothèses, axiomes, présence de `sorry`) et **ce qu'il mesure lui-même** (le mur de l'énumération, les rapports d'exposants). Il ne peut pas affirmer que les théorèmes sont prouvés : les dépôts s'arrêtent sur `sorry` et la preuve (chez anthropics, dans le dossier `proof/`) n'est ni relue ni exécutée ici.

</details>

## Pour aller plus loin

- **Carnet suivant** : [Complexity-04 — Décider sans connaître la suite](Complexity-04-OnlineAlgorithms-Python.ipynb) — la question du coût, mais avec l'information qui arrive dans le désordre.
- **Dans la même série** : [Complexity-02b — le seuil du 3-SAT](Complexity-02b-SAT-PhaseTransition-Python.ipynb) — la densité comme variable de contrôle sur le 3-SAT ; [Complexity-07c — Le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) — le versant empirique de la même vague d'annonces.
- **Carnet voisin d'une autre série** : [SL-10 — Active Automata Learning](../../SymbolicAI/SymbolicLearning/SL-10-ActiveAutomataLearning.ipynb) — apprendre une machine par l'expérimentation, le pendant constructif de ces annonces.
- **Lectures** : le preprint Anthropic *A $2^{o(n)}$ Time Algorithm for 3-SAT* (26 octobre 2026) et le preprint OpenAI *The Exponential Time Hypothesis is False* (25 octobre 2026), tous deux copiés au gisement partagé (dossier `Bibliographie IA/Automata/`) ; Impagliazzo et Paturi (2001), *On the Complexity of k-SAT* (même dossier).